---
# **Práctica 5**: *Fine-tunning y puesta en producción de modelos*
---
### **Alumno**:  Roberto Samuel Sanchez Rosas

In [1]:
#Instalación de dependencias
!pip install transformers datasets evaluate codecarbon accelerate seqeval -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.5/380.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 78.3 MB/s eta 0:00:00


In [2]:
# Importaciones y Activación de CodeCarbon

import os
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
from codecarbon import EmissionsTracker

# Creamos explícitamente el directorio si no existe
output_dir_metrics = "./output_metrics"
if not os.path.exists(output_dir_metrics):
    os.makedirs(output_dir_metrics)
    print(f"Carpeta '{output_dir_metrics}' creada exitosamente.")

# Inicializamos el rastreador ambiental para el reporte de sustentabilidad
tracker = EmissionsTracker(output_dir=output_dir_metrics, log_level="error")
tracker.start()
print("Rastreador CodeCarbon activado para monitorear el Fine-Tuning.")

[codecarbon WARNING @ 01:27:29] Multiple instances of codecarbon are allowed to run at the same time.


Carpeta './output_metrics' creada exitosamente.
Rastreador CodeCarbon activado para monitorear el Fine-Tuning.


In [3]:
# Carga del Dataset de NER y Configuración del Modelo Base

checkpoint = "albert-base-v2"

# CoNLL-2003, dataset estándar para reconocimiento de entidades
raw_datasets = load_dataset("eriktks/conll2003",revision="convert/parquet")

# Extraemos las etiquetas originales del dataset (PER, LOC, ORG, MISC)
label_list = raw_datasets["train"].features["ner_tags"].feature.names

# Creamos diccionarios de mapeo indispensables para la tarea *ForTokenClassification
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print(f"Dataset CoNLL-2003 cargado con éxito.")
print(f"Etiquetas del sistema NER: {label_list}")

conll2003/train/0000.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Dataset CoNLL-2003 cargado con éxito.
Etiquetas del sistema NER: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In [4]:
# Tokenización y Alineación de Etiquetas

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_and_align_labels(examples):
    # Tokenizamos las palabras ya pre-divididas del dataset
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, max_length=512)

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Los tokens especiales de mapeo (como [CLS] o [SEP]) reciben un ID de -100 para ignorarlos en la pérdida
            if word_idx is None:
                label_ids.append(-100)
            # Si el subtoken pertenece a la misma palabra anterior, le asignamos la misma etiqueta o la ignoramos
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Aplicamos la alineación en nuestros conjuntos de datos
tokenized_datasets = raw_datasets.map(tokenize_and_align_labels, batched=True)

# Data Collator específico para Token Classification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
print("Tokenización y alineación de subtokens completada de forma óptima.")


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Tokenización y alineación de subtokens completada de forma óptima.


In [5]:
# Definición de Métricas con Seqeval (Métricas específicas para NER)

# Cargamos la métrica seqeval, diseñada para evaluar secuencias de etiquetas completas
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=-1)

    # Removemos los índices ignorados (-100) para calcular métricas reales
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [6]:
# Configuración de la Arquitectura AlbertForTokenClassification y Trainer

# AutoModel instanciará automáticamente un AlbertForTokenClassification usando los mapeos creados
model = AutoModelForTokenClassification.from_pretrained(
    checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="albert-finetuned-ner",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(4000)),
    eval_dataset=tokenized_datasets["validation"].shuffle(seed=42).select(range(500)),
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Iniciando Fine-Tuning de ALBERT para Clasificación de Tokens (NER)...")
trainer.train()

model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/23 [00:00<?, ?it/s]

AlbertForTokenClassification LOAD REPORT from: albert-base-v2
Key                          | Status     | 
-----------------------------+------------+-
predictions.decoder.bias     | UNEXPECTED | 
albert.pooler.bias           | UNEXPECTED | 
albert.pooler.weight         | UNEXPECTED | 
predictions.dense.bias       | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.bias             | UNEXPECTED | 
predictions.LayerNorm.weight | UNEXPECTED | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Iniciando Fine-Tuning de ALBERT para Clasificación de Tokens (NER)...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.098553,0.854103,0.848943,0.851515,0.971680
2,0.152688,0.076398,0.908748,0.910121,0.909434,0.980746
3,0.152688,0.072990,0.918980,0.925227,0.922093,0.983363


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['albert.embeddings.LayerNorm.weight', 'albert.embeddings.LayerNorm.bias', 'albert.encoder.albert_layer_groups.0.albert_layers.0.attention.LayerNorm.weight', 'albert.encoder.albert_layer_groups.0.albert_layers.0.attention.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['albert.embeddings.LayerNorm.beta', 'albert.embeddings.LayerNorm.gamma', 'albert.encoder.albert_layer_groups.0.albert_layers.0.attention.LayerNorm.beta', 'albert.encoder.albert_layer_groups.0.albert_layers.0.attention.LayerNorm.gamma'].


TrainOutput(global_step=750, training_loss=0.11685791905721028, metrics={'train_runtime': 140.5875, 'train_samples_per_second': 85.356, 'train_steps_per_second': 5.335, 'total_flos': 27243503144352.0, 'train_loss': 0.11685791905721028, 'epoch': 3.0})

In [7]:
# Cierre del Monitoreo y Guardado Local del Modelo

emissions = tracker.stop()
print(f"\n¡Fine-Tuning finalizado de forma exitosa!")
print(f"Emisiones estimadas del entrenamiento individual: {emissions * 1000:.4f} g de CO2")

# Guardamos el modelo final y el tokenizador en una carpeta local
trainer.save_model("./modelo_ner_albert")
tokenizer.save_pretrained("./modelo_ner_albert")
print("Modelo y configuraciones exportadas localmente.")

# Comprimimos para descargar o pasar directamente al Space de Hugging Face
!zip -r modelo_ner_albert.zip ./modelo_ner_albert


¡Fine-Tuning finalizado de forma exitosa!
Emisiones estimadas del entrenamiento individual: 0.6891 g de CO2


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo y configuraciones exportadas localmente.
  adding: modelo_ner_albert/ (stored 0%)
  adding: modelo_ner_albert/tokenizer_config.json (deflated 48%)
  adding: modelo_ner_albert/model.safetensors (deflated 7%)
  adding: modelo_ner_albert/training_args.bin (deflated 53%)
  adding: modelo_ner_albert/config.json (deflated 55%)
  adding: modelo_ner_albert/tokenizer.json (deflated 79%)
